# 02 · Cleaning

Walks through `src/data/clean.py` step by step. All logic lives in `src/`; this notebook only runs it and shows the results.

**Pipeline:** load + join → tech-title filter → text cleaning → dedupe → yearly salary + IQR → location + work type → seniority label → `data/processed/jobs.parquet`

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import clean

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

## 1. Load and join

Skill categories and industries become `|`-joined strings, one row per posting.

In [2]:
raw = clean.load_raw()
print(f"{len(raw):,} postings")
raw[["job_id", "title", "skill_categories", "industries"]].head()

123,849 postings


,job_id,title,skill_categories,industries
0,921716,Marketing Coordinator,Marketing|Sales,Real Estate
1,1829192,Mental Health Therapist/Counselor,Health Care Provider,NaN
2,10998357,Assitant Restaurant Manager,Management|Manufacturing,Restaurants
3,23221523,Senior Elder Law / Trusts and Estates Associate Attorney,Other,Law Practice
4,35982263,Service Technician,Information Technology,Facilities Services


## 2. Tech-title filter

A title is kept if it matches one of the `TECH_INCLUDE` patterns and none of the `TECH_EXCLUDE` patterns (e.g. *sales engineer*, *mechanical*, *data entry*).
Titles are more reliable than descriptions or industries: a nurse at a software company is not a tech job, and a data engineer at a hospital is.

In [3]:
is_tech = clean.is_tech_title(raw["title"])
print(f"Kept {is_tech.sum():,} of {len(raw):,} ({is_tech.mean():.1%})")

pd.DataFrame(
    {
        "kept (random 30)": raw.loc[is_tech, "title"].sample(30, random_state=42).values,
        "dropped (random 30)": raw.loc[~is_tech, "title"].sample(30, random_state=42).values,
    }
)

Kept 10,820 of 123,849 (8.7%)


,kept (random 30),dropped (random 30)
0,Saleforce Developer,Director of Customer Support
1,Senior DevOps Engineer,"Director of Acquisition & Business Development, Qualifying Broker"
2,Business Intelligence and Internal Control Analyst,Lead Services Specialist - Liftra Services Manager
3,C++ Developer,Staff Accountant
4,AI - Machine Learning Architect with Azure,Residence Director
5,Data Science Intern DIN17,Generator Service Technician - Sign on Bonus
6,Data Delivery Lead (AWS),Technician
7,Cybersecurity Engineer,Assistant General Manager
8,Senior Frontend/App Developer,Full Time - Fulfillment Team Lead - Day
9,Staff Software Engineer,Procurement Clerk


Random drops are mostly obviously non-tech. The harder cases are **near misses**: dropped titles that contain tech-sounding words.

In [4]:
near_miss = raw.loc[
    ~is_tech
    & raw["title"].str.contains(
        r"engineer|developer|analyst|technolog|\bit\b|data|architect|comput|cloud|security", case=False
    ),
    "title",
]
near_miss.str.lower().value_counts().head(30).rename("count").to_frame()

,count
title,
business analyst,147
electrical engineer,122
financial analyst,114
project engineer,102
manufacturing engineer,98
quality engineer,92
mechanical engineer,88
board certified behavior analyst (bcba),72
process engineer,69


Deliberately dropped: *business/financial analyst*, *electrical/mechanical/project/quality engineer*, *technician* (maintenance, automotive, pharmacy), *technical writer*, *technical project manager*, and ambiguous titles such as *test engineer*, *application engineer*, *technical lead*.
The ambiguous ones could be software or hardware, so we lean towards precision.

## 3. Run the full pipeline

In [5]:
df, stats = clean.build_dataset(raw)
funnel = pd.Series(
    {
        "raw postings": stats["raw_rows"],
        "tech titles": stats["tech_rows"],
        "after dedupe (and empty descriptions)": stats["deduped_rows"],
    },
    name="rows",
)
funnel.to_frame()

,rows
raw postings,123849
tech titles,10820
after dedupe (and empty descriptions),10040


## 4. Text cleaning

We keep both `description_raw` and `description_clean` (HTML and URLs removed, whitespace collapsed).

In [6]:
has_url = df["description_raw"].str.contains(r"https?://|www\.", na=False)
print(f"Descriptions with URLs: {has_url.mean():.1%}")
example = df.loc[has_url].iloc[0]
url_pos = example["description_raw"].find("http")
start = max(url_pos - 150, 0)
print("RAW:  ", repr(example["description_raw"][start : start + 300]))
print("CLEAN:", repr(clean.clean_text(example["description_raw"][start : start + 300])))

Descriptions with URLs: 13.9%
RAW:   'our mission to transform the talent acquisition industry with innovative technology solutions. We highly encourage you to explore our beta product at https://skillfitai.com/ before submitting your application, allowing you to gain a deeper understanding of our mission and objectives. Become a Freela'
CLEAN: 'our mission to transform the talent acquisition industry with innovative technology solutions. We highly encourage you to explore our beta product at before submitting your application, allowing you to gain a deeper understanding of our mission and objectives. Become a Freela'


## 5. Salary → yearly USD

`HOURLY` × 2080, `WEEKLY` × 52, `BIWEEKLY` × 26, `MONTHLY` × 12. `salary_yearly` is the median salary when given, otherwise the midpoint of min and max.
Outliers outside the IQR fences (Q1 − 1.5·IQR, Q3 + 1.5·IQR) are set to NaN, not dropped, because their text is still useful for the models.

In [7]:
low, high = stats["salary_bounds"]
print(f"IQR fences: ${low:,.0f} to ${high:,.0f}")
print(f"Postings with a usable salary: {stats['salary_rows']:,} ({stats['salary_rows'] / len(df):.1%})")
df.groupby("pay_period", observed=True)["salary_yearly"].describe()[["count", "25%", "50%", "75%"]].round(0)

IQR fences: $6,624 to $260,026
Postings with a usable salary: 2,899 (28.9%)


,count,25%,50%,75%
pay_period,,,,
BIWEEKLY,0.0,NaN,NaN,NaN
HOURLY,925.0,87360.0,124800.0,145600.0
MONTHLY,15.0,76824.0,96426.0,114138.0
YEARLY,1959.0,107950.0,137500.0,171875.0


In [8]:
# Compare outlier rules on the same salaries
import numpy as np

usd = raw.loc[raw["job_id"].isin(df["job_id"])]
usd = usd.loc[usd["currency"].fillna("USD").eq("USD")]
yearly = clean.to_yearly(usd["med_salary"], usd["pay_period"]).fillna(
    (clean.to_yearly(usd["min_salary"], usd["pay_period"]) + clean.to_yearly(usd["max_salary"], usd["pay_period"])) / 2
).dropna()

rules = {
    "IQR k=1.5 (used)": clean.iqr_bounds(yearly, 1.5),
    "IQR k=3": clean.iqr_bounds(yearly, 3),
    "IQR on log(salary)": tuple(np.exp(clean.iqr_bounds(np.log(yearly[yearly > 0]), 1.5))),
}
pd.DataFrame(
    [
        {"rule": name, "low": round(lo), "high": round(hi), "removed": int((~yearly.between(lo, hi)).sum())}
        for name, (lo, hi) in rules.items()
    ]
)

,rule,low,high,removed
0,IQR k=1.5 (used),6624,260026,114
1,IQR k=3,-88402,355051,32
2,IQR on log(salary),49479,339876,159


**Trade-off:** k=1.5 cuts at ≈ \$260k, which also removes some real senior/staff salaries (\$260k–\$480k). The log-IQR rule keeps those, but it cuts everything under ≈ \$49k, which hits hourly internships. k=1.5 is the standard rule and what the plan specified. Either alternative is a one-argument change.

## 6. Location and work type

In [9]:
print(df["location_level"].value_counts().to_string(), end="\n\n")
print(f"State known for {df['state'].notna().mean():.1%} of postings")
df.loc[df["location_level"].eq("metro"), ["location", "city", "state"]].drop_duplicates().head(10)

location_level
city       7340
country    1705
state       560
metro       409
other        26

State known for 82.8% of postings


,location,city,state
0,Los Angeles Metropolitan Area,Los Angeles,CA
10,Greater Philadelphia,Philadelphia,PA
12,San Francisco Bay Area,San Francisco,CA
13,Washington DC-Baltimore Area,Washington,DC
19,Huntsville-Decatur-Albertville Area,Huntsville,AL
81,Atlanta Metropolitan Area,Atlanta,GA
90,New York City Metropolitan Area,New York,NY
165,Dallas-Fort Worth Metroplex,Dallas,TX
291,Greater Seattle Area,Seattle,WA
488,Greater Phoenix Area,Phoenix,AZ


In [10]:
df["state"].value_counts().head(10).rename("postings").to_frame()

,postings
state,
CA,1215
TX,1065
NY,585
VA,505
IL,383
NJ,360
FL,351
GA,341
NC,326


In [11]:
wt = df["work_type"].value_counts()
pd.DataFrame({"count": wt, "pct": (wt / len(df) * 100).round(1)})

,count,pct
work_type,,
on-site,5671,56.5
remote,2529,25.2
hybrid,1840,18.3


`remote_allowed` is the only structured signal (it is either 1 or missing). Hybrid comes from text mentions (excluding "hybrid cloud" etc.), and remote also from the title.
**Caveat:** "on-site" really means "not stated as remote or hybrid".

## 7. Seniority label

In [12]:
print(df["experience_level"].value_counts(dropna=False).to_string(), end="\n\n")
counts = df["seniority"].value_counts()
pd.DataFrame({"count": counts, "pct_of_labeled": (counts / counts.sum() * 100).round(1)})

experience_level
Mid-Senior level    4601
NaN                 2932
Entry level         1570
Associate            666
Director             150
Internship            80
Executive             41



,count,pct_of_labeled
seniority,,
Mid,5267,74.1
Entry,1650,23.2
Senior,191,2.7


The **Senior class is tiny** (~2.7% of labeled rows): Director and Executive roles are rare in tech postings.
Meanwhile LinkedIn's *Mid-Senior level* lumps mid-level and senior engineers together. Alternative mappings:

In [13]:
labeled = df.loc[df["experience_level"].notna()].copy()
level = labeled["experience_level"].astype(str)
senior_title = labeled["title"].str.contains(
    r"\b(?:senior|sr\.?|lead|staff|principal|head|director|vp|chief)\b", case=False
)

options = {
    "A. Spec: Intern+Entry / Assoc+Mid-Sr / Dir+Exec": clean.map_seniority(level),
    "B. Title-assisted: Mid-Sr with senior-ish title -> Senior": clean.map_seniority(level).mask(
        level.eq("Mid-Senior level") & senior_title, "Senior"
    ),
    "C. Associate -> Entry": clean.map_seniority(level, {**clean.SENIORITY_MAP, "Associate": "Entry"}),
    "D. Binary: Entry vs Experienced": clean.map_seniority(level).replace({"Mid": "Experienced", "Senior": "Experienced"}),
}
pd.DataFrame({name: s.value_counts() for name, s in options.items()}).T.fillna(0).astype(int)

experience_level,Entry,Experienced,Mid,Senior
A. Spec: Intern+Entry / Assoc+Mid-Sr / Dir+Exec,1650,0,5267,191
B. Title-assisted: Mid-Sr with senior-ish title -> Senior,1650,0,3276,2182
C. Associate -> Entry,2316,0,4601,191
D. Binary: Entry vs Experienced,1650,5458,0,0


- **A** follows the plan, but Senior is ~190 rows. Class weights help, yet per-class recall on Senior will be noisy.
- **B** gives balanced classes, but the label is then partly *defined by the title*. That makes the "model without the title" experiment in Phase 5 circular for Senior.
- **C** moves Associate (usually 1–3 years of experience) into Entry. That's a small rebalance, and defensible.
- **D** is the most robust statistically, but it loses the three-level story.

The code uses **A** by default (`SENIORITY_MAP`), and switching is a one-line change.

## 8. Save

In [14]:
clean.PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(clean.PROCESSED_PATH, index=False)
size_mb = clean.PROCESSED_PATH.stat().st_size / 1e6
print(f"Saved {len(df):,} rows × {df.shape[1]} columns to data/processed/jobs.parquet ({size_mb:.1f} MB)")
df.dtypes.to_frame("dtype").T

Saved 10,040 rows × 24 columns to data/processed/jobs.parquet (34.7 MB)


,job_id,title,description_raw,description_clean,company_id,company_name,industries,skill_categories,experience_level,seniority,...,state,location_level,pay_period,salary_min,salary_max,salary_yearly,listed_date,original_listed_date,views,applies
dtype,int64,str,str,str,Int64,str,str,str,category,category,...,category,category,category,float32,float32,float32,datetime64[ms],datetime64[ms],float32,float32
